# Optional: Do the carbon atoms tell the same story? Read the carbon spectra yourself

You'll check the notebook 01 peptides again, this time through their carbon atoms.

Hydrogen NMR (1H) shows the protons; carbon NMR (13C) shows the carbon skeleton. In 13C, read which
bands are there, not their size: carbons without hydrogens (such as C=O) give weak signals in quick
runs.

Time: about 30 minutes. Same samples as notebook 01, in 98% acid. Boxes headed **RECORD YOUR
ANSWER** are where you write something; being wrong or unsure is fine, since you compare with the
paper at the end.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/05_read_a_13c_series.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

> **CONNECT TO RAKIURA (optional, recommended)**
>
> **What it does:** ticks your checklist on the step as you go and attaches your figures to it. Nothing is sent until you connect.
>
> **How:** run the cell below, open the link it prints, check the code matches the one printed here, and click Allow. In Colab you do this once per session. To skip the click, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; get the key in Rakiura under Settings > Notebook key).
>
> **Skipping it is fine:** your work is kept on this computer, and you can attach your files on the step page instead.

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = nt.portal.connect(NOTEBOOK, sign_in=True)

### What you need

Ala-Ala has three kinds of carbon, two of each (one per alanine), coloured in the drawing below:

- the carbonyls (C=O): one in the peptide bond, one at the end of the chain
- the alpha carbons: the backbone carbons next to the nitrogen
- the methyl carbons (CH3)

Typical positions in carbon NMR:

| Carbon | Typical position |
|---|---|
| carbonyl (C=O) | near 170 to 180 ppm |
| carbon bonded to nitrogen | about 30 to 65 ppm |
| methyl carbon | about 10 to 30 ppm |

The carbonyl range is from the NMR project Extra on carbon spectra; the other two are general
textbook ranges. A carbon bonded to oxygen or nitrogen sits higher than one bonded only to carbon
and hydrogen.

Longer version: NMR project Extra, Did the molecule really split in two? Count the peaks in a carbon spectrum.

In [ ]:
# Ala-Ala's carbons by kind: orange carbonyls (C=O), blue alpha carbons, green methyl carbons.
fig, ax = plt.subplots(figsize=(7, 3.2))
nt.plot.draw_dipeptide(ax, "Ala", "Ala-Ala: three kinds of carbon",
                       {"C1": ("", "#f6c9a8"), "C2": ("", "#f6c9a8"), "A1": ("", "#b9d7f0"), "A2": ("", "#b9d7f0"),
                        "S1": ("", "#c8e6c3"), "S2": ("", "#c8e6c3")})
plt.show()

> **RECORD YOUR ANSWER: predict where each kind of carbon appears**
>
> **What gets saved:** `MY_PREDICTION`, three positions in ppm, saved with the time.
>
> **Where to look:** the drawing and the table of typical positions above.
>
> **How to fill this in:**
> - `carbonyl_ppm`: the carbonyls (C=O), a number in ppm like `100`
> - `alpha_ppm`: the alpha carbons
> - `methyl_ppm`: Ala-Ala's methyl carbons
> - The cell after it says what the reasoning predicts; you then find the real positions in the stack.

<details><summary>Stuck? Open hint 1</summary>

Start with the carbonyl: the table gives its region.

</details>

<details><summary>Stuck? Open hint 2</summary>

The more oxygen or nitrogen on or next to a carbon, the higher it sits: the carbonyl highest,
the alpha carbon (bonded to nitrogen) next, the methyl lowest.

</details>

In [ ]:
MY_PREDICTION = {
    "carbonyl_ppm": ...,   # a number in ppm: the C=O carbons
    "alpha_ppm": ...,      # a number in ppm: the backbone carbons next to the nitrogen
    "methyl_ppm": ...,     # a number in ppm: Ala-Ala's CH3 carbons
}
nt.record.predict("c13", MY_PREDICTION)

In [ ]:
# Reveal: runs only after you predicted. It checks your reasoning, not this dataset.
if nt.record.revealed("c13"):
    c13_key = nt.answers.load("c13_series", quiet=True)
    nt.checks.check_prediction(MY_PREDICTION, c13_key["prediction"]).show("Your predictions")

## Load and stack

In [ ]:
c13 = {pep: nt.load_series("11223995", pep, acid="K", nucleus="13C") for pep in ("AA", "GG")}
fig_13c, axes = plt.subplots(1, 2, figsize=(14, 5.5))
traps = {}
for ax, (pep, ser) in zip(axes, c13.items()):
    nt.plot.stack(ser, 0, 200, norm=(5, 200), ax=ax, title=f"{nt.io.NAMES[pep]} in 98% acid, 13C")
    traps[pep] = nt.checks.run_traps(ser)
plt.show()

### Trap: missing carbonyls

Look at Gly-Gly near 170 ppm: its carbonyls show in only one spectrum. A band missing from a quick
13C run is not evidence that the carbon is gone.

In [ ]:
for s in c13["GG"]:
    seen = [p["ppm"] for p in nt.peaks.pick(s, 160, 185, 0.05, group_ppm=0.5, ref=(5, 200))]
    print(f"Gly-Gly {s.time_label:9s} carbonyl bands: {seen or 'none seen'}")

## Compare first and last

The next cell lists the peaks in the first and last spectrum of each peptide, and which are at
both times, at the first only, and at the last only. `MIN_HEIGHT` sets the smallest peak it lists.

In [ ]:
MIN_HEIGHT = 0.05
my_peaks, changes = {}, {}
for pep, ser in c13.items():
    my_peaks[pep] = {s.time: nt.peaks.pick(s, 5, 200, MIN_HEIGHT, group_ppm=0.5) for s in (ser[0], ser[-1])}
    for t, pk in my_peaks[pep].items():
        print(nt.io.NAMES[pep], f"{t:7s}", [round(p["ppm"], 1) for p in pk])
    changes[pep] = {k: [round(p, 1) for p in v] for k, v in nt.peaks.compare(*my_peaks[pep].values(), tol=0.5).items()}
    c = {k: ", ".join(f"{p:.1f}" for p in v) or "none" for k, v in changes[pep].items()}
    print(f"   at both times {c['kept']} | first only {c['lost']} | last only {c['new']}")

### What you need

Read Ala-Ala's first (1 day) list. Each number is one carbon band's position in ppm. Use the
table in the first box to say which band is which kind of carbon.

Longer version: NMR project Extra, Did the molecule really split in two? Count the peaks in a carbon spectrum.

> **RECORD YOUR ANSWER: where do Ala-Ala's carbons really sit?**
>
> **What gets saved:** `MY_READING`, three positions in ppm, read off Ala-Ala's 1 day list.
>
> **Where to look:** the Ala-Ala 1 day list printed above, and the left stack.
>
> **How to fill this in:**
> - `carbonyl_ppm`: one of the carbonyl bands, a number like `100.0`
> - `alpha_ppm`: one of the alpha carbon bands
> - `methyl_ppm`: the methyl carbon band
> - The check after it compares your reading with the ranges in this data and with your prediction.

<details><summary>Stuck? Open hint 1</summary>

Sort the 1 day list into three groups using the table of typical positions.

</details>

<details><summary>Stuck? Open hint 2</summary>

The highest values are carbonyls. The values between about 40 and 55 are alpha carbons. The
lowest value is the methyl. Leave out anything that fits none of the three.

</details>

In [ ]:
MY_READING = {
    "carbonyl_ppm": ...,   # a number in ppm, from Ala-Ala's 1 day list
    "alpha_ppm": ...,      # a number in ppm
    "methyl_ppm": ...,     # a number in ppm
}
reading_key = nt.answers.load("c13_series", quiet=True)["reading"]
nt.checks.check_reading(MY_READING, reading_key, MY_PREDICTION).show("Where Ala-Ala's carbons sit at 1 day");

### A filled example: Gly-Gly

Gly-Gly, filled in with the reasons. It shows the format; you do Ala-Ala below.

In [ ]:
GG_EXAMPLE = {
    "call": "survives",        # its carbon bands stay in place, as its hydrogen signals did in notebook 01
    "lost_ppm": [],            # nothing gone (its carbonyls show in only one spectrum: not evidence they are gone)
    "new_ppm": [],             # nothing new
    "kept_ppm": [44.2, 41.3],  # the two CH2 carbons, one per glycine
    "confidence": "medium",    # carbon supports the hydrogen call; quick 13C runs miss weak bands
}

### What you need

- **Lost:** a band at 1 day that is gone at the last time point.
- **New:** a band at the last time point that was not there at 1 day. Expect many small new peaks:
  the other fragment keeps reacting.
- **Kept:** a band at both times.
- Read which bands are there, not their size: peak heights in these carbon spectra do not measure
  amounts.

Longer version: NMR project Extra, Did the molecule really split in two? Count the peaks in a carbon spectrum.

> **RECORD YOUR ANSWER: what changed in Ala-Ala's carbon spectrum?**
>
> **What gets saved:** `AA_LOST_PPM`, `AA_NEW_PPM` and `AA_KEPT_PPM`, three lists of 13C positions in ppm.
>
> **Where to look:** the Ala-Ala lists printed under Compare first and last, and the left stack.
>
> **How to fill this in:**
> - `AA_LOST_PPM`: bands there at 1 day and gone at the last time point, as numbers like `[12.3, 45.6]`
> - `AA_NEW_PPM`: bands at the last time point that were not there at 1 day
> - `AA_KEPT_PPM`: bands there at both times
> - Leave a list as `[]` if there were none. Gly-Gly's record above shows the format.

<details><summary>Stuck? Open hint 1</summary>

Go band by band through the 1 day list: is there still a band within about half a ppm at the last time point?

</details>

<details><summary>Stuck? Open hint 2</summary>

Count the carbonyls first: at 1 day there are two. How many large ones remain at the last
time point?

</details>

In [ ]:
AA_LOST_PPM = []    # bands there at 1 day and gone at the last time point, like [12.3, 45.6]
AA_NEW_PPM = []     # bands at the last time point that were not there at 1 day
AA_KEPT_PPM = []    # bands there at both times

nt.record.write_down("AA_LOST_PPM", AA_LOST_PPM, "Ala-Ala (13C)", "lost_ppm", suggestion=changes["AA"]["lost"])
nt.record.write_down("AA_NEW_PPM", AA_NEW_PPM, "Ala-Ala (13C)", "new_ppm", suggestion=changes["AA"]["new"])
nt.record.write_down("AA_KEPT_PPM", AA_KEPT_PPM, "Ala-Ala (13C)", "kept_ppm", suggestion=changes["AA"]["kept"]);

> **RECORD YOUR ANSWER: your call on Ala-Ala from its carbons**
>
> **What gets saved:** `AA_CALL` and `AA_CONFIDENCE`, two short answers in quotes.
>
> **Where to look:** the lost, new and kept bands you saved above.
>
> **How to fill this in:**
> - `AA_CALL`: `"survives"`, `"breaks down"` or `"unsure"` (unsure is a good answer when the data do not settle it)
> - `AA_CONFIDENCE`: `"low"`, `"medium"` or `"high"`

<details><summary>Stuck? Open hint 1</summary>

Did one of the two carbonyls disappear?

</details>

<details><summary>Stuck? Open hint 2</summary>

If the peptide bond breaks and alaninamide forms, one main carbonyl peak remains. Two at 1 day
and one large one later points to a broken bond.

</details>

In [ ]:
AA_CALL = ""          # "survives", "breaks down" or "unsure"
AA_CONFIDENCE = ""    # "low", "medium" or "high"

nt.record.write_down("AA_CALL", AA_CALL, "Ala-Ala (13C)", "call")
nt.record.write_down("AA_CONFIDENCE", AA_CONFIDENCE, "Ala-Ala (13C)", "confidence");

### Your verdict

Here is your verdict, built from what you recorded above. Change any line above and re-run this
cell to update it. The Gly-Gly row is the filled example.

In [ ]:
MY_VERDICT = nt.record.build_verdict(
    {"AA": {"call": AA_CALL, "lost_ppm": AA_LOST_PPM, "new_ppm": AA_NEW_PPM, "kept_ppm": AA_KEPT_PPM,
            "confidence": AA_CONFIDENCE}},
    names={"AA": "Ala-Ala (13C)"}, examples={"Gly-Gly (13C, example)": GG_EXAMPLE})["AA"]

The next cell compares your call with the published finding (Petkowski et al. 2024, *Scientific
Reports* 14, 17083). If it differs, look again at the carbonyls; your hand-in goes to review as you
wrote it.

In [ ]:
feedback = nt.checks.Report()
if nt.record.revealed("c13"):
    key = nt.answers.load("c13_series")
    c = nt.answers.case(key, "AA")
    feedback = nt.checks.check_verdict(MY_VERDICT, c).show("Your Ala-Ala verdict")

## Hand-in

> **SAVE YOUR HAND-IN: your carbon hand-in**
>
> **What gets saved:** in `handin/`: your 13C stacks and `verdict.json` (your verdict table and the feedback).
>
> **Where to look:** nothing new to fill in: the cell uses what you recorded above.
>
> **How to fill this in:**
> - Run the cell. Nothing is sent for review until you run `nt.portal.submit()`.

<details><summary>Stuck? Open hint 1</summary>

If the figure is missing from the list it prints, run the stack cell, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin("c13-verdict", figures={"stack_13C": fig_13c},
                          verdict={"verdict": MY_VERDICT, "reading": MY_READING, "feedback": [str(f) for f in feedback]},
                          numbers={"AA_call_13C": MY_VERDICT["call"] or "blank"},
                          question="c13", traps=nt.checks.traps_flagged(*traps.values()))
# nt.portal.submit()